In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
loader = TextLoader(r"D:\LLMOps\data\AgenticAI.txt", encoding="utf8")

In [4]:
documents = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=80)

In [6]:
text_chunks = text_splitter.split_documents(documents=documents)

In [7]:
len(text_chunks)

273

In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4087.16it/s]


In [11]:
len(embeddings.embed_query("hi"))

384

In [12]:
vector_store = FAISS.from_documents(text_chunks, embeddings)

In [13]:
retriever = vector_store.as_retriever(kwargs={"k":2})

In [ ]:
retriever.invoke(input="What is agentic AI?")

[Document(id='dfc49ec8-c6e9-422e-a9b6-3d88f99579e0', metadata={'source': 'D:\\LLMOps\\data\\AgenticAI.txt'}, page_content='What is agentic AI?\nWhile there are many different definitions that have been proposed \nfor what constitutes an agentic AI system,1 for the purposes of this'),
 Document(id='61247eb7-ea91-43d3-b827-86dcb53b45d1', metadata={'source': 'D:\\LLMOps\\data\\AgenticAI.txt'}, page_content='and are the basis upon which the agentic AI system analyzes the \nsituation presented to it and plans the steps it should take. They can'),
 Document(id='7cb0783c-aab9-45e0-86ea-74fb65b0a97f', metadata={'source': 'D:\\LLMOps\\data\\AgenticAI.txt'}, page_content='agentic AI system is deployed, the complexity of the goal or outcome \nthe agentic AI system is intending to achieve, and its ability to impact \nthe environment in which it operates.'),
 Document(id='686444dc-423b-4801-8cd5-0461c2848137', metadata={'source': 'D:\\LLMOps\\data\\AgenticAI.txt'}, page_content='for what constitute

In [16]:
from langchain_core.prompts import ChatPromptTemplate

In [18]:
template = """
You are a Question-Answer Assistant. 
use the given retrieved context while giving answer. 
If you donnot answer, simply denied to give answer. 
context: {context} 
question: {question} 
Asnwer: """

In [19]:
prompt = ChatPromptTemplate.from_template(template)

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [22]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [25]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    |llm
    |output_parser
)

In [27]:
rag_chain.invoke("tell me about Agentic AI?")

'Agentic AI refers to artificial intelligence systems that have the ability to analyze situations, plan steps, and execute actions independently. These systems are designed to operate autonomously and can work in concert with other agentic AI systems. They offer numerous benefits for businesses and their customers, including the ability to test various methods, execute actions, and operate independently. The definition of agentic AI can vary, but it generally refers to systems that can analyze situations, plan, and act upon them.'